# Mars heavy-lift nuclear thermal tug — ThunderGraph demo

**Subject:** Notional **NTP** space tug for **Earth–Mars** heavy cargo **orbital transfer** (illustrative numbers only).

**Concept:** **TRISO** barrier narrative, **hydrogen** propellant, **HEU** as **U-235 mass fraction ≥ 0.95** on both the core part and the `requirements.reactor_fuel` package. Hardware is split into **`reactor_core`**, **`propellant_feed`**, **`nozzle`**, **`shadow_shield`**, **`avionics_gnc`**, **`cargo_berthing`**, and **`design_envelope`** (mission capability declaration).

**Patterns:** Program-root **napkin parameters** are structural inputs on the `System`. **`MarsTransferNapkinDesk`** is bound on the owned **`mission_sizing`** part through **`ExternalComputeBinding`** + **`computed_by=`**, where the graph materializes the **`sim_*`** attributes. **`merge_mars_ntp_eval_inputs`** reuses the desk once to fill requirement-package mirrors and reactor/nozzle sizing without copying five derived numbers by hand. Package-level **`parameter` / `attribute` / `constraint`** live under **`requirements.*`**; mission closure uses the deliberate **advanced leaf reqcheck** helpers **`requirement_input`** / **`requirement_accept_expr`** on the **`design_envelope`** allocation path. **`ConfiguredModel.evaluate(inputs=…)`** uses **`ValueSlot` handles** as keys (not **`.stable_id`** strings).

**Source:** `examples/mars_ntp_tug/tug_model.py`, **`examples/mars_ntp_tug/integrations/`** (desk + preset merge), and **`examples/mars_ntp_tug/reporting/`** (`extract_mars_ntp_evaluation_report` + `format_mars_ntp_report`).

**Run:** `uv sync --group dev` then run cells, or headless:  
`uv run --group dev python -m jupyter nbconvert --execute --inplace notebooks/mars_ntp_nuclear_tug.ipynb`


## Requirements vs structure

| Tag | Requirement / package | Allocate target part |
|-----|------------------------|------------------------|
| R-FUEL | `requirements.reactor_fuel.*` | `reactor_core` |
| R-TH | `requirements.thermal_hydraulic.*` | `reactor_core` |
| R-THRUST | `requirements.propulsion.*` | `nozzle` |
| R-SHIELD | `requirements.shielding.*` | `shadow_shield` |
| R-MISSION | `requirements.mission.*` (executable) | `design_envelope` |
| R-SAFE | `requirements.safety_policy.*` (text + citation) | `reactor_core` |


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

def _ensure_pkg_root() -> None:
    cwd = Path.cwd()
    for base in (cwd, cwd.parent, cwd / "thundergraph-model", cwd.parent / "thundergraph-model"):
        if (base / "tg_model").is_dir() and (base / "examples" / "mars_ntp_tug").is_dir():
            root = str(base.resolve())
            if root not in sys.path:
                sys.path.insert(0, root)
            return
    raise RuntimeError("Could not find thundergraph-model package root (tg_model + examples/mars_ntp_tug).")

_ensure_pkg_root()

from examples.mars_ntp_tug.integrations import (
    merge_mars_ntp_eval_inputs,
    reference_hardware_overrides,
    reference_napkin_assumptions,
    run_mars_transfer_napkin_desk,
)
from examples.mars_ntp_tug.tug_model import MarsNuclearTug, reset_ntp_types
from examples.mars_ntp_tug.reporting import extract_mars_ntp_evaluation_report, format_mars_ntp_report
from tg_model.execution.configured_model import instantiate


In [2]:
reset_ntp_types()
cm = instantiate(MarsNuclearTug)

napkin = reference_napkin_assumptions()
hardware = reference_hardware_overrides()
desk_preview = run_mars_transfer_napkin_desk(napkin)
print("Napkin desk (same algebra the graph runs under computed_by=):")
for key in sorted(desk_preview):
    print(f"  {key}: {desk_preview[key]}")

inputs = merge_mars_ntp_eval_inputs(cm, napkin=napkin, hardware=hardware)
result = cm.evaluate(inputs=inputs)
report = extract_mars_ntp_evaluation_report(cm, result)
print(format_mars_ntp_report(report))
print(f"\nEvaluator passed: {result.passed}")
if result.failures:
    for line in result.failures:
        print("  failure:", line)
print("Reference edges:", len(cm.references))


Napkin desk (same algebra the graph runs under computed_by=):
  hydrogen_mass_flow_kg_s: 44.72547060218143 kg/s
  propellant_kg: 91070.11536568787 kg
  thermal_power_mw: 504.266806826109 MW
  vacuum_thrust_kn: 394.74633265279425 kN
  wet_start_kg: 196355.7245949429 kg
Mars NTP tug (notional) — evaluation snapshot
Verdict (all constraints + mission reqcheck rows): PASS
  Evaluator completed without engine failures: True
  Mission reqcheck (requirement_accept_expr) all passed: True (count: 2)
Thesis (read this once)
Program-root **napkin parameters** feed
:class:`~examples.mars_ntp_tug.integrations.adapters.MarsTransferNapkinDesk`
via ``computed_by=`` (same integration pattern as the cargo jet mission
desk). The desk produces **sim_*** attributes (propellant, wet mass,
thrust floor, mass flow, thermal power). **Mission** closure still uses
``requirement_accept_expr`` on ``requirements.mission``; **package**
parameters for thermal-hydraulic and propulsion are mirrored from the
desk in ``m